# Colab setup with persistent Google Drive results

This notebook is a thin wrapper around the repository's existing `src` and `scripts`.

**Persistence/resume behavior:**
- All experiment outputs are written **directly to Google Drive**, not `/content`.
- Each completed run is therefore persisted immediately.
- The repository scripts are run with `SKIP_EXISTING=1`, so after a Colab disconnect you can reconnect, rerun the setup/upload cells, and rerun the experiment cell. Already-completed runs are skipped and execution continues from the remaining runs.
- The three datasets themselves are still uploaded to the Colab session each time, as requested.

The notebooks cover the three paper datasets: TelecomTS, MIT Supercloud GPU utilization, and ESNet `r-nova-fd-01`.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, subprocess, sys, shutil, pandas as pd, json

# Mount Google Drive first so results are persistent from the first completed run onward.
drive.mount('/content/drive')

# Change this only if you want a different Drive folder.
DRIVE_ROOT = Path('/content/drive/MyDrive/SystemsExcursionForecastability')
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

REPO_URL = 'PASTE_GIT_REPOSITORY_URL_HERE'
REPO_DIR = Path('/content/SystemsExcursionForecastability')

if not REPO_DIR.exists():
    if REPO_URL == 'PASTE_GIT_REPOSITORY_URL_HERE':
        raise ValueError('Set REPO_URL before running this cell.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

print('Repository ready:', REPO_DIR)
print('Persistent results root:', DRIVE_RESULTS)

In [ ]:
# Upload ALL THREE prepared CSVs to this Colab runtime.
# They do not need to be in Drive; only results are persisted there.
uploaded = files.upload()

DATA_DIR = REPO_DIR / 'data' / 'processed'
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_uploaded(*needles):
    matches = [n for n in uploaded if all(x.lower() in n.lower() for x in needles)]
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one uploaded file matching {needles}; found {matches}; '
            f'uploaded={list(uploaded)}'
        )
    return matches[0]

telecom_upload = find_uploaded('telecom')
mit_upload = find_uploaded('mit')
esnet_upload = find_uploaded('esnet')

TELECOM_CSV = DATA_DIR / 'telecomts.csv'
MIT_CSV = DATA_DIR / 'mit_supercloud_gpu_utilization_gpu_pct.csv'
ESNET_CSV = DATA_DIR / 'esnet_r-nova-fd-01_delta.csv'

for src, dst in [
    (telecom_upload, TELECOM_CSV),
    (mit_upload, MIT_CSV),
    (esnet_upload, ESNET_CSV),
]:
    shutil.copy2(src, dst)

print('Uploaded datasets:')
print('  TelecomTS:', TELECOM_CSV)
print('  MIT:', MIT_CSV)
print('  ESNet:', ESNET_CSV)

In [ ]:
TELECOM_SERIES = 'telecomts::normal/stationary/Zone_B/no_congestion/File::TX_Bytes'
MIT_SERIES = 'mit_gpu::67691045781943-r7343737-n685852::utilization_gpu_pct'
ESNET_SERIES = 'esnet::r-nova-fd-01::ifHCOutOctets'

TELECOM_SIGNALS = ','.join([
    'TX_Bytes', 'RX_Bytes', 'Estimated_UL_Buffer',
    'PRB_Utilization_DL', 'PRB_Utilization_UL'
])
ESNET_SIGNALS = ','.join([
    'ifHCOutOctets', 'ifHCOutUcastPkts', 'ifHCOutMulticastPkts',
    'ifHCOutBroadcastPkts', 'ifHCInOctets', 'ifHCInUcastPkts',
    'ifHCInMulticastPkts', 'ifHCInBroadcastPkts'
])

# Trivial baseline experiment

To guarantee partial persistence, this wrapper invokes `scripts/run_trivial_baselines.py` **once per dataset** rather than passing all three datasets in a single manifest.

Each dataset therefore gets its own Drive subdirectory and result files immediately. If Colab disconnects after TelecomTS or MIT finishes, those results are already safely in Drive.

In [ ]:
BASELINE_ROOT = DRIVE_RESULTS / 'baselines'
MANIFEST_DIR = DRIVE_RESULTS / 'baseline_manifests'
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

baseline_jobs = [
    ('telecom', TELECOM_CSV, TELECOM_SERIES),
    ('mit', MIT_CSV, MIT_SERIES),
    ('esnet', ESNET_CSV, ESNET_SERIES),
]

In [ ]:
for name, data_path, series_id in baseline_jobs:
    print(f'\n===== {name} =====', flush=True)
    dataset_output = BASELINE_ROOT / name
    dataset_output.mkdir(parents=True, exist_ok=True)
    marker = dataset_output / '_COMPLETED.txt'

    # Dataset-level baseline jobs are inexpensive relative to model training. Skip a
    # fully completed dataset on resume. If no completion marker exists, rerun only
    # this dataset rather than restarting all three.
    if marker.exists():
        print('SKIP completed dataset:', name)
        continue

    manifest = MANIFEST_DIR / f'{name}.csv'
    pd.DataFrame([{
        'data': str(data_path),
        'series_id': series_id,
    }]).to_csv(manifest, index=False)
    print('Manifest persisted:', manifest)

    cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts' / 'run_trivial_baselines.py'),
        str(manifest),
        '--output-dir', str(dataset_output),
        '--random-trials', '1000',
        '--random-seed', '1',
    ]
    print('>>>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

    marker.write_text('completed\n')
    print('Persisted completion marker:', marker)

## Resume behavior

After a disconnect, rerun the notebook from the top. Completed dataset directories containing `_COMPLETED.txt` are skipped. Only the interrupted/unstarted dataset jobs run again.